In [ ]:
import pandas as pd
import pickle

with open("nearest_categories.pkl", "rb") as f:
    nearest_categories: dict[str, pd.DataFrame] = pickle.load(f)

for category, df in nearest_categories.items():
    display(f"category: {category}", df.describe(), df.head())


In [ ]:
import pandana as pdna

network = pdna.Network.from_hdf5("denmark.backup")

In [ ]:
# Check what nodes in the network has all categories within max_dist
dist_df = pd.concat(
    [df["dist"] for df in nearest_categories.values()], 
    axis=1
)

max_dist = 1600
nodes = network.nodes_df

nodes["score"] = (dist_df <= max_dist).sum(axis=1)
valid_nodes = nodes[nodes["score"] > 0]

display(valid_nodes.head(), len(valid_nodes.index))

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    data=valid_nodes["score"],
    geometry=gpd.points_from_xy(valid_nodes["x"], valid_nodes["y"]),
    crs=4326
)

gdf.to_file("accessibility-full.json", driver="GeoJSON")